# 13주차 2차 수업 — CSV에서 SQLite로 (Colab)

저장 계층만 바꾼다. 테스트는 모두 통과시킨다.

이 노트북은 슬라이드의 코드를 **위에서 아래로 순서대로** 실행하도록 만들었다. 필요한 파일은 노트북이 직접 만들기 때문에 업로드할 것이 없다.

| 부 | 내용 | 슬라이드 |
|---|---|---|
| 0 | Colab 기본: 셀 실행, `!`와 `%cd` | — |
| 1 | 준비: `market.db` 만들기 | 3 |
| 2 | 행 바꾸기: UPDATE와 DELETE | 5–7 |
| 3 | 요약하기: COUNT, SUM, GROUP BY | 8–11 |
| 4 | 트랜잭션: 전부 아니면 전무 | 12–15 |
| 5 | 저장 계층 교체하기 | 17–23 |
| 6 | 실습 A–D | 24–26 |
| 7 | 제출 | 27 |

## 0. Colab 기본

- **셀 실행**: 셀을 선택하고 `Shift + Enter`를 누르거나 왼쪽의 ▶ 단추를 누른다.
- **순서가 중요하다**: 뒤의 셀은 앞의 셀이 만든 변수(`con`, `count`)와 파일을 쓴다. 중간부터 실행하면 `NameError`가 난다. 그럴 때는 메뉴에서 **런타임 → 모두 실행** 또는 위에서부터 다시 실행한다.
- **코드 셀과 텍스트 셀**: 지금 읽는 것은 텍스트 셀이다. 회색 상자가 코드 셀이다.

아래 셀을 실행해 Python과 SQLite 버전을 확인한다. `sqlite3`는 Python에 들어 있어서 설치할 것이 없다.

In [ ]:
import sys
import sqlite3

print("Python", sys.version.split()[0])
print("SQLite", sqlite3.sqlite_version)

### 터미널 명령 쓰기

슬라이드의 `bash` 상자는 터미널 명령이다. Colab에서는 두 가지만 기억한다.

| 슬라이드 | Colab | 뜻 |
|---|---|---|
| `python3 main.py` | `!python3 main.py` | 줄 앞에 `!`를 붙이면 터미널 명령이다 |
| `cd 폴더` | `%cd 폴더` | 폴더 이동은 `%cd`. `!cd`는 그 줄에서만 유효하다 |

Colab의 작업 폴더는 `/content`다. 이 수업의 파일은 모두 `/content/week13` 아래에 둔다.

In [ ]:
!mkdir -p /content/week13/sqlite_lab
%cd /content/week13/sqlite_lab
!pwd

> **주의: 파일은 사라진다.** Colab은 한동안 쓰지 않으면 런타임을 초기화하고 `/content`의 파일을 지운다. 이 노트북을 위에서부터 다시 실행하면 파일은 다시 만들어지지만, **내가 고친 코드는 셀 안에 있어야** 남는다. 그래서 이 노트북에서는 파일을 셀에서 고친다(5부에서 설명한다). 먼저 **파일 → Drive에 사본 저장**으로 내 사본을 만들자.

## 1. 준비: market.db 만들기

1차시에 만든 `sqlite_lab` 폴더의 파일 세 개를 다시 만든다. 첫 줄이 `%%writefile 이름`인 셀은 실행하면 **셀의 내용을 그 이름의 파일로 저장**한다.

- `schema.sql`: 규칙이 있는 `quotes` 테이블.
- `prices.csv`: 다섯 종목의 행.
- `create_db.py`: 둘을 읽어 `market.db`를 만드는 스크립트.

In [ ]:
%%writefile schema.sql
CREATE TABLE IF NOT EXISTS quotes (
    date   TEXT NOT NULL,
    code   TEXT NOT NULL,
    name   TEXT NOT NULL,
    market TEXT NOT NULL,
    previous_close INTEGER NOT NULL
        CHECK (previous_close > 0),
    close  INTEGER NOT NULL
        CHECK (close > 0),
    volume INTEGER NOT NULL
        CHECK (volume >= 0),
    PRIMARY KEY (date, code)
);

In [ ]:
%%writefile prices.csv
date,code,name,market,previous_close,close,volume
2026-11-05,005930,삼성전자,KOSPI,70000,71400,1000000
2026-11-05,000660,SK하이닉스,KOSPI,200000,198000,500000
2026-11-05,035420,NAVER,KOSPI,200000,200000,300000
2026-11-05,247540,에코프로비엠,KOSDAQ,100000,103000,200000
2026-11-05,035900,JYP Ent.,KOSDAQ,50000,49000,100000

In [ ]:
%%writefile create_db.py
"""Create market.db from schema.sql and load prices.csv. Safe to run again."""
import csv
import sqlite3
from pathlib import Path

BASE = Path(__file__).resolve().parent
INSERT = """
    INSERT INTO quotes
    VALUES (?, ?, ?, ?, ?, ?, ?)
"""


def main():
    schema = (BASE / "schema.sql").read_text(encoding="utf-8")
    with open(BASE / "prices.csv", encoding="utf-8", newline="") as f:
        rows = [tuple(r.values()) for r in csv.DictReader(f)]
    con = sqlite3.connect(BASE / "market.db")
    con.executescript(schema)
    con.execute("DELETE FROM quotes")
    con.executemany(INSERT, rows)
    con.commit()
    count = con.execute("SELECT COUNT(*) FROM quotes").fetchone()[0]
    con.close()
    print(f"market.db ready: {count} rows")


if __name__ == "__main__":
    main()

스크립트를 실행한다. 여러 번 실행해도 안전하다. 항상 다섯 행으로 돌아간다.

In [ ]:
!python3 create_db.py

예상 출력

```text
market.db ready: 5 rows
```

### 복습: market.db의 다섯 행 (슬라이드 3)

- connect, execute, commit, close.
- 값은 `?` 자리표시자로 전달한다.
- SELECT는 묻기만 하고 테이블을 바꾸지 않는다.

이 셀이 만드는 `con`과 `count`를 2~4부의 모든 셀이 쓴다.

In [ ]:
import sqlite3

con = sqlite3.connect("market.db")
count = "SELECT COUNT(*) FROM quotes"
print(con.execute(count).fetchone())

query = "SELECT code, close FROM quotes"
for row in con.execute(query):
    print(row)

예상 출력

```text
(5,)
('005930', 71400)
('000660', 198000)
('035420', 200000)
('247540', 103000)
('035900', 49000)
```

> **다시 시작하는 셀.** 실습하다가 테이블이 엉망이 되면 아래 셀을 실행한다. 연결을 닫고, 데이터베이스를 다시 만들고, 다시 연결한다. 지금은 실행하지 않아도 된다.

In [ ]:
con.close()
!python3 create_db.py
con = sqlite3.connect("market.db")

예상 출력

```text
market.db ready: 5 rows
```

## 2. 행 바꾸기: UPDATE와 DELETE

한 줄로 테이블을 망칠 수 있는 두 문장.

- UPDATE는 이미 있는 행의 값을 바꾼다. DELETE는 행을 지운다.
- 둘 다 WHERE가 고른 모든 행에 적용된다. **WHERE가 없으면 모든 행이다.**

### WHERE가 있는 UPDATE (슬라이드 6)

- SET에 열 이름과 새 값을 쓴다.
- `rowcount`는 바뀐 행의 수를 알려 준다.

> **함정**: 틀린 코드는 오류가 아니다. 조용히 0행을 바꾼다.

In [ ]:
UPDATE = """
    UPDATE quotes SET close = ?
    WHERE code = ?
"""
cur = con.execute(UPDATE, (72000, "005930"))
print(cur.rowcount)

cur = con.execute(UPDATE, (72000, "999999"))
print(cur.rowcount)

예상 출력

```text
1
0
```

### 잊어버린 WHERE (슬라이드 7)

- `DELETE FROM quotes`에는 WHERE가 없다. 모든 행이 대상이다.
- 아직 커밋하지 않았으므로 `rollback()`으로 되돌린다.
- 앞 셀의 72000 수정도 함께 되돌아간다.

> **팁**: 같은 WHERE로 SELECT를 먼저 실행해 보고, 그다음에 지운다.

In [ ]:
cur = con.execute("DELETE FROM quotes")
print(cur.rowcount)
print(con.execute(count).fetchone())

con.rollback()
print(con.execute(count).fetchone())

예상 출력

```text
5
(0,)
(5,)
```

72000 수정도 되돌아갔는지 확인한다. 삼성전자의 종가는 다시 71400이어야 한다.

In [ ]:
query = "SELECT close FROM quotes WHERE code = ?"
print(con.execute(query, ("005930",)).fetchone())

예상 출력

```text
(71400,)
```

## 3. 요약하기: 세고 묶기

행이 아니라 답을 요청한다.

- 집계 함수는 여러 행을 값 하나로 만든다.
- GROUP BY는 그룹마다 답 하나를 준다.

### COUNT, SUM, MAX, AVG (슬라이드 9)

> **예측**: 이 쿼리는 튜플을 몇 개 돌려줄까? 실행하기 전에 답을 정한다.

In [ ]:
query = """
    SELECT COUNT(*), SUM(volume),
           MAX(close), AVG(close)
    FROM quotes
"""
print(con.execute(query).fetchone())

예상 출력

```text
(5, 2100000, 200000, 124280.0)
```

### GROUP BY: 시장마다 한 행 (슬라이드 10)

- market이 같은 행들이 한 그룹이 된다.
- SELECT의 열은 그룹 열이거나 집계 함수여야 한다.

> **함정**: `GROUP BY market`에 `SELECT name`이라면 어느 name일까? 요청하지 않는다.

In [ ]:
query = """
    SELECT market, COUNT(*), SUM(volume)
    FROM quotes
    GROUP BY market
    ORDER BY market
"""
for row in con.execute(query):
    print(row)

예상 출력

```text
('KOSDAQ', 2, 300000)
('KOSPI', 3, 1800000)
```

### 규칙은 어디에 둘까? (슬라이드 11)

- 비교식은 참이면 1, 거짓이면 0이다. 비교식의 SUM은 조건이 참인 행의 수다.
- 답은 `summarize()`와 같다: 상승 2, 하락 2, 보합 1.

> **연결**: 규칙의 집은 한 곳이다. 우리는 테스트가 있는 `logic.py`. SQL은 행을 가져온다.

In [ ]:
query = """
    SELECT SUM(close > previous_close),
           SUM(close < previous_close),
           SUM(close = previous_close)
    FROM quotes
"""
print(con.execute(query).fetchone())

예상 출력

```text
(2, 2, 1)
```

## 4. 트랜잭션: 전부 아니면 전무

함께 성공하거나 함께 취소되는 변경의 묶음.

- 트랜잭션은 첫 변경과 함께 시작된다.
- `commit()`은 전부를 확정하고 `rollback()`은 전부를 버린다. 그 중간 상태는 없다.

### 새 행 둘, 그중 하나는 불량 (슬라이드 14)

둘째 행은 close가 0이다. CHECK가 거절해야 한다.

> **예측**: 실패한 뒤 테이블의 행은 몇 개일까? 5, 6, 7 중에서 고르자.

In [ ]:
INSERT = """
    INSERT INTO quotes
    VALUES (?, ?, ?, ?, ?, ?, ?)
"""
new_rows = [
    ("2026-11-06", "005930", "삼성전자",
     "KOSPI", 71400, 72100, 900000),
    ("2026-11-06", "000660", "SK하이닉스",
     "KOSPI", 198000, 0, 450000),
]

### with con: 커밋 또는 롤백 (슬라이드 15)

- 블록이 정상적으로 끝나면 커밋한다.
- 블록에서 예외가 나면 롤백하고 예외를 다시 전달한다.
- 1행은 들어갔다가 2행과 함께 취소됐다.

> **함정**: `with con`은 연결을 닫지 않는다. 트랜잭션만 끝낸다.

In [ ]:
try:
    with con:
        con.executemany(INSERT, new_rows)
except sqlite3.IntegrityError as error:
    print(error)

print(con.execute(count).fetchone())

예상 출력

```text
CHECK constraint failed: close > 0
(5,)
```

**휴식 전 질문.** `storage.py`에서 어떤 함수가 바뀌어야 하고, 어떤 함수는 그대로일까?

## 5. 저장 계층 교체하기

호출하는 쪽과의 약속은 그대로, 안쪽 장치만 새로.

### 행이 logic.py가 받는 dict가 된다 (슬라이드 19)

- 키는 열 이름이고 테이블의 열 순서다.
- code는 여전히 글자이고 앞자리 0이 남아 있다.
- 가격과 거래량은 int다. `int()`를 부를 필요가 없다.

In [ ]:
con.row_factory = sqlite3.Row
query = "SELECT * FROM quotes ORDER BY code"
quotes = [dict(r) for r in con.execute(query)]

for key, value in quotes[0].items():
    print(f"{key:<15}{value!r}")

예상 출력

```text
date           '2026-11-05'
code           '000660'
name           'SK하이닉스'
market         'KOSPI'
previous_close 200000
close          198000
volume         500000
```

`sqlite_lab`에서 할 일은 끝났다. 연결을 닫는다.

In [ ]:
con.close()

### 교체한 MVP의 파일 만들기

이제 11~12주차의 한국 증권시장 MVP로 간다. 폴더를 만들고 이동한 뒤, 파일을 하나씩 만든다.

| 파일 | 12주차와 비교 |
|---|---|
| `logic.py`, `report.py` | **한 줄도 바뀌지 않았다** |
| `storage.py` | 연결(connection)을 중심으로 다시 썼다 |
| `main.py` | 연결을 열고 넘기는 몇 줄만 바뀌었다 |
| `test_market.py` | 기존 12개 + 저장 계층 6개 = 18개 |

In [ ]:
!mkdir -p /content/week13/korea_market_sqlite/data
%cd /content/week13/korea_market_sqlite

데이터와 스키마. `invalid-zero.csv`는 첫 행의 전일 종가가 0인 잘못된 파일이다.

In [ ]:
%%writefile schema.sql
CREATE TABLE IF NOT EXISTS quotes (
    date   TEXT NOT NULL,
    code   TEXT NOT NULL,
    name   TEXT NOT NULL,
    market TEXT NOT NULL,
    previous_close INTEGER NOT NULL
        CHECK (previous_close > 0),
    close  INTEGER NOT NULL
        CHECK (close > 0),
    volume INTEGER NOT NULL
        CHECK (volume >= 0),
    PRIMARY KEY (date, code)
);

In [ ]:
%%writefile data/prices.csv
date,code,name,market,previous_close,close,volume
2026-11-05,005930,삼성전자,KOSPI,70000,71400,1000000
2026-11-05,000660,SK하이닉스,KOSPI,200000,198000,500000
2026-11-05,035420,NAVER,KOSPI,200000,200000,300000
2026-11-05,247540,에코프로비엠,KOSDAQ,100000,103000,200000
2026-11-05,035900,JYP Ent.,KOSDAQ,50000,49000,100000

In [ ]:
%%writefile data/invalid-zero.csv
date,code,name,market,previous_close,close,volume
2026-11-05,005930,삼성전자,KOSPI,0,71400,1000000
2026-11-05,000660,SK하이닉스,KOSPI,200000,198000,500000
2026-11-05,035420,NAVER,KOSPI,200000,200000,300000
2026-11-05,247540,에코프로비엠,KOSDAQ,100000,103000,200000
2026-11-05,035900,JYP Ent.,KOSDAQ,50000,49000,100000

In [ ]:
%%writefile data/empty.csv
date,code,name,market,previous_close,close,volume

In [ ]:
%%writefile .gitignore
*.db
__pycache__/

규칙(`logic.py`)과 보고서(`report.py`). SQLite를 전혀 모른다. `import sqlite3`가 없다는 것을 확인하자.

In [ ]:
%%writefile logic.py
"""Pure rules for a single-date classroom snapshot; prices are integer KRW."""
from datetime import date

FIELDS = ["date", "code", "name", "market", "previous_close", "close", "volume"]
MARKETS = ("ALL", "KOSPI", "KOSDAQ")


def parse_row(row):
    """Return a typed quote; raise ValueError for any invalid field. No I/O."""
    if set(row) != set(FIELDS) or any(not isinstance(v, str) for v in row.values()):
        raise ValueError("each row needs exactly seven fields")
    quote = {key: value.strip() for key, value in row.items()}
    day = quote["date"]
    if date.fromisoformat(day).isoformat() != day:
        raise ValueError("date must be YYYY-MM-DD")
    code = quote["code"]
    if len(code) != 6 or not code.isascii() or not code.isdigit():
        raise ValueError("code must be six ASCII digits")
    if not quote["name"] or quote["market"] not in MARKETS[1:]:
        raise ValueError("name and KOSPI/KOSDAQ market are required")
    for field in ("previous_close", "close", "volume"):
        text = quote[field]
        if not text.isascii() or not text.isdigit():
            raise ValueError(f"{field} must be a whole number")
        quote[field] = int(text)
        minimum = 0 if field == "volume" else 1
        if quote[field] < minimum:
            raise ValueError(f"{field} must be >= {minimum}")
    return quote


def change_pct(previous_close, close):
    """Return unrounded daily percent change; both prices must be positive."""
    if previous_close <= 0 or close <= 0:
        raise ValueError("prices must be positive")
    return (close - previous_close) / previous_close * 100


def analyze(quotes, market):
    """Validate snapshot identity; filter and sort by change desc, then code asc.

    Return fresh quote dicts with change_pct. Empty input/selection returns [].
    Mixed dates, duplicate codes or an unknown market raise ValueError.
    quotes must already contain typed, validated values from parse_row().
    """
    if market not in MARKETS:
        raise ValueError("market must be ALL, KOSPI or KOSDAQ")
    if len({q["date"] for q in quotes}) > 1:
        raise ValueError("snapshot must contain one date")
    codes = [q["code"] for q in quotes]
    if len(set(codes)) != len(codes):
        raise ValueError("duplicate code in snapshot")
    rows = [dict(q, change_pct=change_pct(q["previous_close"], q["close"]))
            for q in quotes if market == "ALL" or q["market"] == market]
    return sorted(rows, key=lambda q: (-q["change_pct"], q["code"]))


def summarize(rows):
    """Return sample count, up/down/flat counts and equal-weight mean (or None)."""
    count = len(rows)
    return {"count": count,
            "up": sum(q["change_pct"] > 0 for q in rows),
            "down": sum(q["change_pct"] < 0 for q in rows),
            "flat": sum(q["change_pct"] == 0 for q in rows),
            "mean": sum(q["change_pct"] for q in rows) / count if count else None}

In [ ]:
%%writefile report.py
"""Pure HTML formatting. Escape all external text before interpolation."""
from html import escape


def render_html(rows, summary, market, snapshot_date):
    """Return a complete HTML page; round percentages only when displaying."""
    body = []
    for q in rows:
        change = q["change_pct"]
        direction = "up" if change > 0 else "down" if change < 0 else "flat"
        body.append(f'<tr><td>{escape(q["code"])}</td>'
                    f'<td>{escape(q["name"])}</td><td>{escape(q["market"])}</td>'
                    f'<td>{q["close"]:,}</td><td class="{direction}">{change:+.2f}%</td>'
                    f'<td>{q["volume"]:,}</td></tr>')
    mean = "N/A" if summary["mean"] is None else f'{summary["mean"]:+.2f}%'
    table_body = "".join(body) or '<tr><td colspan="6">표시할 종목이 없습니다.</td></tr>'
    return f'''<!doctype html>
<html lang="ko"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>한국 주식시장 분석 실습</title>
<style>
body {{font-family:system-ui,sans-serif;background:#f3f5f9;color:#17243c;margin:0}}
main {{max-width:1040px;margin:48px auto;padding:24px}}
h1 {{font-size:2rem}} .meta {{color:#53627b}} .summary {{background:#17243c;color:white;padding:24px;border-radius:12px}}
.scroll {{overflow-x:auto}} table {{border-collapse:collapse;width:100%;background:white;margin-top:24px}}
caption {{text-align:left;padding:12px 0;font-weight:bold}}
th,td {{padding:14px;text-align:right;border-bottom:1px solid #dde3ec;white-space:nowrap}}
th:nth-child(-n+3),td:nth-child(-n+3) {{text-align:left}}
.up {{color:#b12a38}} .down {{color:#245bbb}} .flat {{color:#53627b}}
footer {{margin-top:24px;color:#53627b;line-height:1.6}}
</style></head><body><main>
<p class="meta">PYTHON / WEEK 11 / CLASSROOM DATA</p>
<h1>한국 주식시장 분석</h1>
<p>기준일: {escape(snapshot_date)} · 시장: {escape(market)} · 출처: 수업용 가상 CSV</p>
<section class="summary" aria-label="표본 요약">
종목 {summary["count"]}개 · 상승 {summary["up"]} · 하락 {summary["down"]} · 보합 {summary["flat"]}
<p>표본 단순평균 등락률 <strong>{mean}</strong></p></section>
<div class="scroll"><table><caption>일간 등락률 내림차순 / 동률은 종목코드 오름차순</caption>
<thead><tr><th scope="col">코드</th><th scope="col">종목</th><th scope="col">시장</th>
<th scope="col">종가 (원)</th><th scope="col">등락률</th><th scope="col">거래량 (주)</th></tr></thead>
<tbody>{table_body}</tbody></table></div>
<footer>교육용 가상 데이터입니다. 실제 시세 또는 현재 상장 종목 목록이 아닙니다.<br>
표본의 단순평균이며 KOSPI/KOSDAQ 지수 수익률이 아닙니다. 투자 판단용 정보가 아닙니다.</footer>
</main></body></html>'''

### storage.py — SQLite를 아는 유일한 모듈

아래 셀에서 네 이름을 찾아보자.

- `connect()`와 `init_db()`가 파일을 열고 준비한다.
- `replace_quotes()`가 검증된 스냅숏을 **트랜잭션 하나**로 저장한다(슬라이드 20).
- `load_quotes()`는 전과 같은 dict 목록을 돌려준다(슬라이드 18).
- `read_csv()`는 예전 `load_quotes(path)`다. 가져오기용으로 남겼다.

> **실습 B에서 고치는 셀이 이 셀이다.** 셀에서 코드를 고치고 다시 실행하면 파일이 새로 저장된다.

In [ ]:
%%writefile storage.py
"""SQLite storage, CSV import and atomic HTML output; all I/O lives here."""
import csv
import os
from pathlib import Path
import sqlite3
import tempfile
from logic import FIELDS, parse_row

SCHEMA = Path(__file__).resolve().parent / "schema.sql"
INSERT = """
    INSERT INTO quotes
    VALUES (?, ?, ?, ?, ?, ?, ?)
"""


def connect(path):
    """Open the database (":memory:" in tests) with rows readable by name."""
    if str(path) != ":memory:":
        Path(path).parent.mkdir(parents=True, exist_ok=True)
    con = sqlite3.connect(path)
    con.row_factory = sqlite3.Row
    return con


def init_db(con):
    """Create the quotes table if it is missing. Safe to call every run."""
    con.executescript(SCHEMA.read_text(encoding="utf-8"))


def replace_quotes(con, quotes):
    """Replace the whole snapshot in one transaction: all rows or none.

    quotes are typed dicts from parse_row(). A constraint failure raises
    sqlite3.IntegrityError and leaves the previous rows in place.
    """
    rows = [tuple(q[f] for f in FIELDS) for q in quotes]
    with con:
        con.execute("DELETE FROM quotes")
        con.executemany(INSERT, rows)


def load_quotes(con):
    """Return every stored quote as a dict, in code order. Empty table -> []."""
    query = "SELECT * FROM quotes ORDER BY code"
    return [dict(row) for row in con.execute(query)]


def read_csv(path):
    """Read UTF-8(-BOM) CSV; reject malformed header/rows with ValueError.

    Missing or unreadable files raise OSError. Header-only files return [].
    (This was load_quotes() in the week 11 MVP.)
    """
    with open(path, encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != FIELDS:
            raise ValueError("CSV header must be " + ",".join(FIELDS))
        quotes = []
        for number, row in enumerate(reader, start=2):
            try:
                quotes.append(parse_row(row))
            except ValueError as error:
                raise ValueError(f"CSV row {number}: {error}") from error
    return quotes


def write_html(path, html):
    """Replace output only after a complete UTF-8 write; raise OSError on failure."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8",
                                         dir=path.parent, delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(html)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

`main.py`는 연결을 열고, 필요하면 CSV를 가져오고, 보고서를 쓴다.

In [ ]:
%%writefile main.py
"""Generate a static page from Korean stock classroom data stored in SQLite."""
import argparse
from contextlib import closing
from pathlib import Path
import sqlite3
from logic import MARKETS, analyze, summarize
from storage import (connect, init_db, load_quotes, read_csv,
                     replace_quotes, write_html)
from report import render_html

BASE = Path(__file__).resolve().parent


def main(argv=None):
    """Return 0 after successful generation, 1 on data/file errors; CLI errors use 2.

    --import-csv validates a CSV and replaces the stored snapshot first.
    A failed import changes neither the database nor the old HTML.
    """
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--db", type=Path, default=BASE / "data/market.db")
    parser.add_argument("--import-csv", type=Path, metavar="CSV")
    parser.add_argument("--output", type=Path, default=BASE / "docs/index.html")
    parser.add_argument("--market", choices=MARKETS, default="ALL")
    args = parser.parse_args(argv)
    try:
        with closing(connect(args.db)) as con:
            init_db(con)
            if args.import_csv:
                replace_quotes(con, read_csv(args.import_csv))
            quotes = load_quotes(con)
        rows = analyze(quotes, args.market)
        summary = summarize(rows)
        day = quotes[0]["date"] if quotes else "데이터 없음"
        html = render_html(rows, summary, args.market, day)
        write_html(args.output, html)
    except (OSError, ValueError, UnicodeError, sqlite3.Error) as error:
        print(f"Report not generated: {error}")
        return 1
    print(f"Generated {args.output} ({len(rows)} stocks)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

### 메모리 데이터베이스로 테스트하기 (슬라이드 21)

`":memory:"`는 파일이 없는 데이터베이스다. `StorageTests`의 `setUp`이 테스트마다 빈 테이블을 새로 만든다. 그래서 테스트가 실제 `market.db`를 망가뜨릴 수 없다.

In [ ]:
%%writefile test_market.py
"""REQ-linked checks: python3 -m unittest -v."""
import contextlib
import io
import sqlite3
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch

from logic import analyze, change_pct, parse_row, summarize
from main import BASE, main
from report import render_html
from storage import (connect, init_db, load_quotes, read_csv,
                     replace_quotes, write_html)


class MarketTests(unittest.TestCase):
    def setUp(self):
        self.quotes = read_csv(BASE / 'data/prices.csv')

    def test_01_schema_and_leading_zero(self):
        self.assertEqual(self.quotes[0]['code'], '005930')
        self.assertEqual(self.quotes[0]['close'], 71400)

    def test_02_changes(self):
        for previous, close, expected in [(70000, 71400, 2), (200000, 198000, -1), (100, 100, 0)]:
            self.assertAlmostEqual(change_pct(previous, close), expected)

    def test_03_reject_invalid_fields(self):
        row = {k: str(v) for k, v in self.quotes[0].items()}
        for field, bad in [('previous_close', '0'), ('close', '-1'), ('volume', '-1'),
                           ('close', 'NaN'), ('volume', '1.5'), ('code', '5930'),
                           ('date', '2026-02-30'), ('name', ''), ('market', 'OTHER')]:
            with self.subTest(field=field, bad=bad), self.assertRaises(ValueError):
                parse_row(dict(row, **{field: bad}))

    def test_04_filter_and_sort(self):
        rows = analyze(self.quotes, 'KOSPI')
        self.assertEqual([q['code'] for q in rows], ['005930', '035420', '000660'])
        self.assertNotIn('change_pct', self.quotes[0])
        tied = [dict(self.quotes[0], code='000002'), dict(self.quotes[0], code='000001')]
        self.assertEqual(analyze(tied, 'ALL')[0]['code'], '000001')
        with self.assertRaises(ValueError):
            analyze(self.quotes, 'OTHER')

    def test_05_sample_summary(self):
        summary = summarize(analyze(self.quotes, 'ALL'))
        self.assertEqual((summary['count'], summary['up'], summary['down'], summary['flat']), (5, 2, 2, 1))
        self.assertAlmostEqual(summary['mean'], 0.4)

    def test_06_empty_data(self):
        for rows in [analyze([], 'ALL'), analyze(self.quotes[:1], 'KOSDAQ')]:
            summary = summarize(rows)
            self.assertIsNone(summary['mean'])
            html = render_html(rows, summary, 'KOSDAQ', '데이터 없음')
            self.assertIn('표시할 종목이 없습니다', html)
            self.assertIn('N/A', html)

    def test_07_snapshot_identity(self):
        for rows in [self.quotes + [self.quotes[0]],
                     [self.quotes[0], dict(self.quotes[1], date='2026-11-04')]]:
            with self.assertRaises(ValueError):
                analyze(rows, 'ALL')

    def test_08_html_escape(self):
        rows = analyze([dict(self.quotes[0], name='<script>alert(1)</script>')], 'ALL')
        html = render_html(rows, summarize(rows), 'ALL', '2026-11-05')
        self.assertNotIn('<script>', html)
        self.assertIn('&lt;script&gt;', html)
        self.assertIn('+2.00%', html)
        self.assertIn('71,400', html)
        self.assertIn('005930', html)

    def test_09_failed_import_preserves_report_and_rows(self):
        with tempfile.TemporaryDirectory() as folder:
            root = Path(folder)
            db = root / 'market.db'
            output = root / 'index.html'
            with contextlib.redirect_stdout(io.StringIO()):
                self.assertEqual(main(['--db', str(db), '--output', str(output),
                                       '--import-csv', str(BASE / 'data/prices.csv')]), 0)
            output.write_text('OLD', encoding='utf-8')
            bad = root / 'bad.csv'
            bad.write_text('wrong,header\n1,2\n', encoding='utf-8')
            for source in [root / 'missing.csv', bad, BASE / 'data/invalid-zero.csv']:
                with contextlib.redirect_stdout(io.StringIO()):
                    self.assertEqual(main(['--db', str(db), '--output', str(output),
                                           '--import-csv', str(source)]), 1)
                self.assertEqual(output.read_text(), 'OLD')
                with contextlib.closing(connect(db)) as con:
                    self.assertEqual(len(load_quotes(con)), 5)

    def test_10_atomic_write_failure(self):
        with tempfile.TemporaryDirectory() as folder:
            output = Path(folder) / 'index.html'
            output.write_text('OLD')
            with patch('storage.os.replace', side_effect=OSError('simulated failure')):
                with self.assertRaises(OSError):
                    write_html(output, 'NEW')
            self.assertEqual(output.read_text(), 'OLD')
            self.assertEqual(list(Path(folder).iterdir()), [output])

    def test_11_end_to_end(self):
        with tempfile.TemporaryDirectory() as folder:
            output = Path(folder) / 'docs/index.html'
            db = Path(folder) / 'data/market.db'
            with contextlib.redirect_stdout(io.StringIO()):
                self.assertEqual(main(['--db', str(db), '--import-csv', str(BASE / 'data/prices.csv'),
                                       '--market', 'KOSPI', '--output', str(output)]), 0)
                # A second run reads the stored rows; no CSV is needed.
                self.assertEqual(main(['--db', str(db), '--market', 'KOSPI', '--output', str(output)]), 0)
            html = output.read_text(encoding='utf-8')
            self.assertIn('삼성전자', html)
            self.assertNotIn('에코프로비엠', html)
            self.assertIn('종목 3개', html)

    def test_12_csv_boundaries(self):
        header = 'date,code,name,market,previous_close,close,volume\n'
        with tempfile.TemporaryDirectory() as folder:
            source = Path(folder) / 'prices.csv'
            source.write_text('\ufeff' + header, encoding='utf-8')
            self.assertEqual(read_csv(source), [])
            for tail in ['2026-11-05,005930,삼성전자,KOSPI,70000,71400\n',
                         '2026-11-05,005930,삼성전자,KOSPI,70000,71400,0,extra\n']:
                source.write_text(header + tail, encoding='utf-8')
                with self.assertRaises(ValueError):
                    read_csv(source)


class StorageTests(unittest.TestCase):
    """Week 13: the storage layer, checked against an in-memory database."""

    def setUp(self):
        self.con = connect(':memory:')
        init_db(self.con)
        self.quotes = read_csv(BASE / 'data/prices.csv')

    def tearDown(self):
        self.con.close()

    def test_13_empty_table_is_empty_list(self):
        self.assertEqual(load_quotes(self.con), [])

    def test_14_round_trip_keeps_types(self):
        replace_quotes(self.con, self.quotes)
        stored = load_quotes(self.con)
        self.assertEqual(stored, sorted(self.quotes, key=lambda q: q['code']))
        self.assertEqual(stored[0]['code'], '000660')
        self.assertIsInstance(stored[0]['close'], int)

    def test_15_replace_does_not_append(self):
        replace_quotes(self.con, self.quotes)
        replace_quotes(self.con, self.quotes[:2])
        self.assertEqual(len(load_quotes(self.con)), 2)

    def test_16_failed_replace_keeps_old_rows(self):
        replace_quotes(self.con, self.quotes)
        for bad in [self.quotes[:1] * 2,                      # duplicate key
                    [dict(self.quotes[0], close=0)],          # CHECK
                    [dict(self.quotes[0], name=None)]]:       # NOT NULL
            with self.subTest(bad=bad), self.assertRaises(sqlite3.IntegrityError):
                replace_quotes(self.con, bad)
            self.assertEqual(len(load_quotes(self.con)), 5)

    def test_17_same_analysis_from_csv_and_database(self):
        replace_quotes(self.con, self.quotes)
        self.assertEqual(analyze(load_quotes(self.con), 'ALL'), analyze(self.quotes, 'ALL'))

    def test_18_init_db_twice_keeps_rows(self):
        replace_quotes(self.con, self.quotes)
        init_db(self.con)
        self.assertEqual(len(load_quotes(self.con)), 5)


if __name__ == '__main__':
    unittest.main()

### 교체한 MVP 실행하기 (슬라이드 22)

먼저 테스트. 마지막 줄에 `Ran 18 tests`와 `OK`가 나와야 한다.

In [ ]:
!python3 -m unittest

CSV를 한 번 가져온다. 파일을 검증한 뒤 스냅숏을 교체하고 보고서를 쓴다.

In [ ]:
!python3 main.py --import-csv data/prices.csv

이제 CSV 없이 `market.db`만 읽어 보고서를 쓴다. KOSPI는 3종목이다.

In [ ]:
!python3 main.py --market KOSPI

만들어진 보고서를 노트북 안에서 본다.

In [ ]:
from IPython.display import HTML

HTML(open("docs/index.html", encoding="utf-8").read())

잘못된 CSV를 가져온다. `Report not generated`가 나와야 하고, 테이블도 기존 HTML도 바뀌지 않아야 한다.

In [ ]:
!python3 main.py --import-csv data/invalid-zero.csv

정말 그대로인지 확인한다. 저장된 행은 여전히 5개다.

In [ ]:
import sqlite3

check = sqlite3.connect("data/market.db")
print(check.execute("SELECT COUNT(*) FROM quotes").fetchone())
check.close()

예상 출력

```text
(5,)
```

## 6. 실습: 내 저장소 교체하기

34분. 테스트가 안전망이다. 실습 양식(`storage-swap.md`)에 답을 적으면서 진행한다.

### A. 통과에서 시작 (4분)

고치기 전에 테스트 18개를 실행한다. `OK`가 나와야 한다.

In [ ]:
!python3 -m unittest

### B. 시장 필터를 SQL로 (12분)

`load_quotes(con, market="ALL")`이 되도록 고친다.

- `market`이 `"ALL"`이면 모든 행, 아니면 `WHERE market = ?`로 고른 행을 돌려준다.
- `load_quotes(con)`처럼 `market` 없이 부르는 기존 코드가 그대로 동작해야 한다.

**하는 법**: 위로 올라가 `%%writefile storage.py` 셀에서 `load_quotes()`를 고치고 그 셀을 다시 실행한다. 그다음 아래 셀로 확인한다. 고치기 전에는 `TypeError`가 난다.

In [ ]:
!python3 -c "from storage import *; con = connect(':memory:'); init_db(con); replace_quotes(con, read_csv('data/prices.csv')); print(len(load_quotes(con, 'KOSPI')), len(load_quotes(con, 'ALL')), len(load_quotes(con)))"

다 고쳤다면 `3 5 5`가 나온다. 기존 테스트 18개도 다시 실행해 본다.

In [ ]:
!python3 -m unittest

### C. `:memory:` 테스트 (8분)

아래 셀의 테스트 세 개를 완성한다. `self.skipTest(...)` 줄을 지우고 `assertEqual`을 쓴다.

| 입력 | 기대 결과 |
|---|---|
| `load_quotes(con, "KOSPI")` | 3행, 모두 KOSPI |
| `load_quotes(con, "ALL")` | 5행 |
| `load_quotes(con)` | 5행 |

In [ ]:
%%writefile test_lab.py
"""Lab C: market filter checked against an in-memory database."""
import unittest

from main import BASE
from storage import connect, init_db, load_quotes, read_csv, replace_quotes


class MarketFilterTests(unittest.TestCase):
    def setUp(self):
        self.con = connect(":memory:")
        init_db(self.con)
        replace_quotes(self.con, read_csv(BASE / "data/prices.csv"))

    def tearDown(self):
        self.con.close()

    def test_kospi_has_three_rows(self):
        self.skipTest("TODO: 이 줄을 지우고 테스트를 쓴다")

    def test_all_has_five_rows(self):
        self.skipTest("TODO: 이 줄을 지우고 테스트를 쓴다")

    def test_default_has_five_rows(self):
        self.skipTest("TODO: 이 줄을 지우고 테스트를 쓴다")

내 테스트만 자세히 실행한다. 완성하기 전에는 `skipped`로 표시된다.

In [ ]:
!python3 -m unittest -v test_lab

전체를 실행한다. 기존 18개에 내 테스트 3개를 더해 21개가 `OK`여야 한다.

In [ ]:
!python3 -m unittest

### D. 내 프로젝트의 테이블 (10분)

CREATE TABLE을 쓰기 전에 네 가지에 답한다.

- 내 프로젝트에서 한 행은 무엇인가?
- 어떤 열이 있고, 각각 TEXT, INTEGER, REAL 중 무엇인가?
- 어떤 열이 기본 키가 되는가?
- 테이블이 거절해야 하는 값은 무엇인가?

아래 셀에서 내 테이블을 `:memory:`에 만들어 본다. 예시를 내 프로젝트의 테이블로 바꾼다. 파일이 없는 데이터베이스이므로 몇 번이든 다시 실행할 수 있다.

> **팁**: 모든 프로젝트에 데이터베이스가 필요하지는 않다. CSV로 충분하면 그 이유를 적는다.

In [ ]:
import sqlite3

MY_SCHEMA = """
    CREATE TABLE IF NOT EXISTS books (
        isbn  TEXT PRIMARY KEY,
        title TEXT NOT NULL,
        price INTEGER NOT NULL CHECK (price >= 0)
    );
"""
my = sqlite3.connect(":memory:")
my.executescript(MY_SCHEMA)

with my:
    my.execute("INSERT INTO books VALUES (?, ?, ?)",
               ("9788900000001", "파이썬 입문", 18000))
print(my.execute("SELECT * FROM books").fetchall())

try:
    with my:
        my.execute("INSERT INTO books VALUES (?, ?, ?)",
                   ("9788900000002", "잘못된 가격", -1))
except sqlite3.IntegrityError as error:
    print(error)
my.close()

예상 출력

```text
[('9788900000001', '파이썬 입문', 18000)]
CHECK constraint failed: price >= 0
```

## 7. 제출과 다음 주

증거가 먼저, 그다음이 풀 리퀘스트다.

- SQLite를 쓰는 `storage.py`와 unittest 출력.
- 새로 쓴 테스트와, 그 테스트가 잡아내는 실패 하나.
- 내 프로젝트의 테이블 설계: 열, 자료형, 키.
- 브랜치 하나, PR 하나. `.db` 파일은 넣지 않는다.

### 데이터베이스가 아니라 스키마를 커밋한다 (슬라이드 23)

`market.db`는 결과물이고 `schema.sql`은 소스 코드다. `.gitignore`에 `*.db`가 있는지 확인한다.

In [ ]:
!cat .gitignore

예상 출력

```text
*.db
__pycache__/
```

### 내 파일 내려받기

Colab에서는 GitHub 인증이 번거롭다. 아래 셀로 프로젝트 폴더를 zip으로 내려받은 뒤, 내 컴퓨터의 저장소에 풀어 커밋한다. `.db` 파일은 zip에 넣지 않는다.

```bash
git status --short
git add .gitignore schema.sql storage.py test_lab.py
git commit -m "Store quotes in SQLite"
```

In [ ]:
!cd /content/week13 && rm -f week13-session2.zip && zip -r -q week13-session2.zip korea_market_sqlite -x "*.db" "*__pycache__*"

from google.colab import files
files.download("/content/week13/week13-session2.zip")

> **연결**: 14주차에는 Flask로 이 행을 브라우저에 보여 주고 폼으로 새 행을 넣는다.